<a href="https://colab.research.google.com/github/devpedroandrade/classificacao-artistas-spotify/blob/main/Analise_Identidade_Sonora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

print("--- 1. PREPARAÇÃO DOS DADOS ---")
df = pd.read_csv('dataset_mata64.csv')
df['artista'] = df['artista'].replace({'Bjork': 'Björk'})

min_musicas = df['artista'].value_counts().min()
df_balanceado = df.groupby('artista').sample(n=min_musicas, random_state=42)

df_balanceado.to_csv('dataset_mata64_balanceado.csv', index=False)
print("Distribuição Balanceada:")
print(df_balanceado['artista'].value_counts())

print("\n--- 2. MODELO GAUSSIAN NAIVE BAYES ---")
X = df_balanceado[['danceability', 'energy', 'acousticness', 'valence']]
y = df_balanceado['artista']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modelo_nb = GaussianNB()
modelo_nb.fit(X_train, y_train)
y_pred_nb = modelo_nb.predict(X_test)

acuracia_nb = accuracy_score(y_test, y_pred_nb)
print(f"✅ Acurácia do Naive Bayes: {acuracia_nb * 100:.2f}%\n")

plt.figure(figsize=(8, 5))
sns.heatmap(confusion_matrix(y_test, y_pred_nb, labels=modelo_nb.classes_),
            annot=True, fmt='d', cmap='Blues', xticklabels=modelo_nb.classes_, yticklabels=modelo_nb.classes_)
plt.title('Matriz de Confusão - Gaussian Naive Bayes')
plt.ylabel('Real')
plt.xlabel('Previsto')
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestClassifier

print("--- 3. MODELO RANDOM FOREST ---")
df_bal = pd.read_csv('dataset_mata64_balanceado.csv')
X = df_bal[['danceability', 'energy', 'acousticness', 'valence']]
y = df_bal['artista']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modelo_rf = RandomForestClassifier(n_estimators=100, random_state=42)
modelo_rf.fit(X_train, y_train)
y_pred_rf = modelo_rf.predict(X_test)

acuracia_rf = accuracy_score(y_test, y_pred_rf)
print(f"✅ Acurácia do Random Forest: {acuracia_rf * 100:.2f}%\n")

plt.figure(figsize=(8, 5))
sns.heatmap(confusion_matrix(y_test, y_pred_rf, labels=modelo_rf.classes_),
            annot=True, fmt='d', cmap='Greens', xticklabels=modelo_rf.classes_, yticklabels=modelo_rf.classes_)
plt.title('Matriz de Confusão - Random Forest')
plt.ylabel('Real')
plt.xlabel('Previsto')
plt.show()

print("\n--- 4. IMPORTÂNCIA DAS VARIÁVEIS (FEATURE IMPORTANCE) ---")
importancias = modelo_rf.feature_importances_

plt.figure(figsize=(8, 4))
sns.barplot(x=importancias, y=X.columns, hue=X.columns, palette='viridis', legend=False)
plt.title('Importância dos Atributos Acústicos')
plt.xlabel('Relevância')
plt.ylabel('Atributo')
plt.show()